In [1]:
%%capture output
#install Wikipedia API
!pip3 install wikipedia-api


In [2]:
import numpy as np
import pandas as pd
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

pd.options.display.max_columns = 100

import nltk
# nltk.download('omw-1.4')
nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

import wikipediaapi


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
%%capture
!python -m textblob.download_corpora


In [4]:
sentence_1 = 'Jen is a good student.'
sentence_2 = 'Jen is also a great guitarist.'
sentence_3 = 'Good students can sometimes be good guitarists'


# Data Cleaning
We want to singularize guitarists and students.

In [5]:
sentence_3_tb = TextBlob(sentence_3) # Make a textblob so that we can singularize the word
sentence_3_singular = [x.singularize() for x in sentence_3_tb.words] # Singularize each word in the text
sentence_3_clean = ' '.join(sentence_3_singular) # Join it together into a single string
sentence_3_clean


'Good student can sometime be good guitarist'

## Bag of Words Using CountVectorizer

In [6]:
# Perform the count transformation
vectorizer = CountVectorizer(stop_words='english')
bow_vec = vectorizer.fit_transform([sentence_1, sentence_2, sentence_3_clean])
bow_vec


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 9 stored elements and shape (3, 5)>

In [7]:
bow_vec.toarray()


array([[1, 0, 0, 1, 1],
       [0, 1, 1, 1, 0],
       [2, 0, 1, 0, 1]])

In [8]:
# Print out results in a data frame
sent_df = pd.DataFrame(bow_vec.toarray(), columns = vectorizer.get_feature_names_out())
sent_df


,good,great,guitarist,jen,student
0,1,0,0,1,1
1,0,1,1,1,0
2,2,0,1,0,1


In [9]:
[sentence_1, sentence_2, sentence_3_clean]


['Jen is a good student.',
 'Jen is also a great guitarist.',
 'Good student can sometime be good guitarist']

### Your Turn
1. Write 4 sentences of your choice.
2. Run the `CountVectorizer` on your sentences.
3. Print the results in a data frame.

In [10]:
# Solution 1
my_sents = [
    "It was the best of times, it was the worst of times.",
    "Call me Ishmael.",
    "To be, or not to be.  That is the question.",
    "We do not choose these things because they are easy."
]
my_sents

['It was the best of times, it was the worst of times.',
 'Call me Ishmael.',
 'To be, or not to be.  That is the question.',
 'We do not choose these things because they are easy.']

In [11]:
# Solution 2
my_vectorizer = CountVectorizer(stop_words='english')
my_bow_vec = my_vectorizer.fit_transform( my_sents )
my_bow_vec.toarray()


array([[1, 0, 0, 0, 0, 0, 2, 1],
       [0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0],
       [0, 1, 1, 0, 0, 1, 0, 0]])

In [12]:
# Solution 3
my_sent_df = pd.DataFrame(my_bow_vec.toarray(), columns = my_vectorizer.get_feature_names_out())
my_sent_df


,best,choose,easy,ishmael,question,things,times,worst
0,1,0,0,0,0,0,2,1
1,0,0,0,1,0,0,0,0
2,0,0,0,0,1,0,0,0
3,0,1,1,0,0,1,0,0


## TF-IDF

In [13]:
# Perform the TF-IDF transformation - Option 1 (TfidfVectorizer)
tf_idf_vec = TfidfVectorizer(stop_words = 'english')
tf_idf_jen = tf_idf_vec.fit_transform([sentence_1, sentence_2, sentence_3_clean])
tf_idf_jen


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>

In [14]:
print(sentence_1)
print(sentence_2)
print(sentence_3_clean)


Jen is a good student.
Jen is also a great guitarist.
Good student can sometime be good guitarist


In [15]:
# Print out results in a dataframe
tf_df = pd.DataFrame(tf_idf_jen.toarray(), columns = tf_idf_vec.get_feature_names_out())
tf_df.shape


(3, 5)

In [16]:
tf_df


,good,great,guitarist,jen,student
0,0.577350,0.000000,0.000000,0.577350,0.577350
1,0.000000,0.680919,0.517856,0.517856,0.000000
2,0.816497,0.000000,0.408248,0.000000,0.408248


In [17]:
# Perform the TF-IDF transformation - Option 2 (CountVectorizer + TfidfTransformer - better for large datasets)
tf_idf_tran = TfidfTransformer()
tf_idf_jen = tf_idf_tran.fit_transform(bow_vec)
tf_idf_jen

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9 stored elements and shape (3, 5)>

In [18]:
# Print out results in a dataframe
tf_df = pd.DataFrame(tf_idf_jen.toarray(), columns = vectorizer.get_feature_names_out())
tf_df


,good,great,guitarist,jen,student
0,0.577350,0.000000,0.000000,0.577350,0.577350
1,0.000000,0.680919,0.517856,0.517856,0.000000
2,0.816497,0.000000,0.408248,0.000000,0.408248


In [19]:
# Get a data frame with the TF-IDF values sorted for document 0
df = pd.DataFrame(tf_idf_jen[0].T.todense(), index=tf_idf_vec.get_feature_names_out(), columns=["TF-IDF"])
df = df.sort_values('TF-IDF', ascending=False)
df


,TF-IDF
good,0.57735
student,0.57735
jen,0.57735
guitarist,0.00000
great,0.00000


In [20]:
tf_df.transpose()[0].sort_values(ascending = False)

,0
good,0.57735
student,0.57735
jen,0.57735
guitarist,0.00000
great,0.00000


### Your Turn
1. Use the `TfidfTransformer` to transform the bag of words matrix you created above to TF-IDF.
2. Print out the results in a data frame.

These settings might be helpful.

In [21]:
np.set_printoptions(linewidth=200)
np.set_printoptions(precision=6, formatter={'all': lambda x: f'{x:.4f}'})
my_bow_vec.toarray()

array([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 2.0000, 1.0000],
       [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 1.0000, 1.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000]])

In [22]:
# Solution 1
my_tf_idf_tran = TfidfTransformer()
my_tf_idf_jen = my_tf_idf_tran.fit_transform(my_bow_vec)
my_tf_idf_jen


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 8 stored elements and shape (4, 8)>

In [23]:
my_tf_idf_jen.toarray()


array([[0.4082, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.8165, 0.4082],
       [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000],
       [0.0000, 0.5774, 0.5774, 0.0000, 0.0000, 0.5774, 0.0000, 0.0000]])

In [24]:
# Solution 2
my_tf_df = pd.DataFrame(my_tf_idf_jen.toarray(), columns = my_vectorizer.get_feature_names_out())
my_tf_df


,best,choose,easy,ishmael,question,things,times,worst
0,0.408248,0.00000,0.00000,0.0,0.0,0.00000,0.816497,0.408248
1,0.000000,0.00000,0.00000,1.0,0.0,0.00000,0.000000,0.000000
2,0.000000,0.00000,0.00000,0.0,1.0,0.00000,0.000000,0.000000
3,0.000000,0.57735,0.57735,0.0,0.0,0.57735,0.000000,0.000000


# Another Example - Using Wikipedia API

In [25]:
# Pull out the popcorn page from wikipedia - https://en.wikipedia.org/wiki/Popcorn
topic = 'popcorn'
wikip = wikipediaapi.Wikipedia(user_agent = 'foobar')
page_ex = wikip.page(topic)
wiki_text = page_ex.text
wiki_text


'Popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated. The term also refers to the snack food produced by the expansion. It is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the Americas. It is commonly eaten salted, buttered, sweetened, or with artificial flavorings.\nA popcorn kernel\'s strong hull contains the seed\'s hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated. Pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool.\nSome strains of corn (botanically classified as Zea mays) are cultivated specifically as popping corns. The Zea mays variety everta, a special kind of flint corn, is the most common of these. Popcorn is one of six major types of corn, alongside dent corn, flint corn, pod corn, flour corn, and sweet corn.\n

In [26]:
! curl -L 'https://en.wikipedia.org/wiki/Popcorn'


<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-underline-clientpref-browser" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Popcorn - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main

In [31]:
page_ex.categories


{'Category:All articles with unsourced statements': Category:All articles with unsourced statements (lang: en, variant: None, id: ??, ns: 14),
 'Category:American snack foods': Category:American snack foods (lang: en, variant: None, id: ??, ns: 14),
 'Category:Articles tagged with the inline citation overkill template from October 2025': Category:Articles tagged with the inline citation overkill template from October 2025 (lang: en, variant: None, id: ??, ns: 14),
 "Category:Articles with 'species' microformats": Category:Articles with 'species' microformats (lang: en, variant: None, id: ??, ns: 14),
 'Category:Articles with limited geographic scope from November 2026': Category:Articles with limited geographic scope from November 2026 (lang: en, variant: None, id: ??, ns: 14),
 'Category:Articles with short description': Category:Articles with short description (lang: en, variant: None, id: ??, ns: 14),
 'Category:Articles with unsourced statements from January 2021': Category:Article

In [32]:
type(wiki_text)


str

In [33]:
print(wiki_text[:4000])


Popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated. The term also refers to the snack food produced by the expansion. It is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the Americas. It is commonly eaten salted, buttered, sweetened, or with artificial flavorings.
A popcorn kernel's strong hull contains the seed's hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated. Pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool.
Some strains of corn (botanically classified as Zea mays) are cultivated specifically as popping corns. The Zea mays variety everta, a special kind of flint corn, is the most common of these. Popcorn is one of six major types of corn, alongside dent corn, flint corn, pod corn, flour corn, and sweet corn.

Histo

### Clean the text - version 1

Using string replace.



In [34]:
# Replace newline chars with spaces before doing any processing. Strip the ' and "s" from possessives
wiki_text_clean = (
    wiki_text
    .lower()
    .replace("\n"," ")
    .replace("'s",'')
    .replace("'",'')
)
wiki_text_clean


'popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated. the term also refers to the snack food produced by the expansion. it is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the americas. it is commonly eaten salted, buttered, sweetened, or with artificial flavorings. a popcorn kernel strong hull contains the seed hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated. pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool. some strains of corn (botanically classified as zea mays) are cultivated specifically as popping corns. the zea mays variety everta, a special kind of flint corn, is the most common of these. popcorn is one of six major types of corn, alongside dent corn, flint corn, pod corn, flour corn, and sweet corn.  history 

In [35]:
print(wiki_text_clean[:4000])


popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated. the term also refers to the snack food produced by the expansion. it is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the americas. it is commonly eaten salted, buttered, sweetened, or with artificial flavorings. a popcorn kernel strong hull contains the seed hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated. pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool. some strains of corn (botanically classified as zea mays) are cultivated specifically as popping corns. the zea mays variety everta, a special kind of flint corn, is the most common of these. popcorn is one of six major types of corn, alongside dent corn, flint corn, pod corn, flour corn, and sweet corn.  history c

### Clean the text - version 2

Using a for..loop and string replace.




In [36]:
wiki_text_clean = wiki_text.lower()
for c in ["\n", "'s", "'", "  "]:
  wiki_text_clean = wiki_text_clean.replace(c," ")
wiki_text_clean


'popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated. the term also refers to the snack food produced by the expansion. it is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the americas. it is commonly eaten salted, buttered, sweetened, or with artificial flavorings. a popcorn kernel strong hull contains the seed hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated. pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool. some strains of corn (botanically classified as zea mays) are cultivated specifically as popping corns. the zea mays variety everta, a special kind of flint corn, is the most common of these. popcorn is one of six major types of corn, alongside dent corn, flint corn, pod corn, flour corn, and sweet corn. history c

### Clean the text - version 3

Using a regular expression.


In [37]:
import re

pat = re.compile(r"(\n|'s|'| )+")
wiki_text_clean = re.sub(pat,' ', wiki_text.lower())
wiki_text_clean


'popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated. the term also refers to the snack food produced by the expansion. it is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the americas. it is commonly eaten salted, buttered, sweetened, or with artificial flavorings. a popcorn kernel strong hull contains the seed hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated. pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool. some strains of corn (botanically classified as zea mays) are cultivated specifically as popping corns. the zea mays variety everta, a special kind of flint corn, is the most common of these. popcorn is one of six major types of corn, alongside dent corn, flint corn, pod corn, flour corn, and sweet corn. history c

### Moving on ...


In [38]:
# Break up single string into separate sentences
wiki_blob = TextBlob(wiki_text_clean)
len(wiki_blob.sentences)


128

In [44]:
# Only look at first 20 sentences
my_sentences = wiki_blob.sentences[:]
my_sentences


[Sentence("popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated."),
 Sentence("the term also refers to the snack food produced by the expansion."),
 Sentence("it is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the americas."),
 Sentence("it is commonly eaten salted, buttered, sweetened, or with artificial flavorings."),
 Sentence("a popcorn kernel strong hull contains the seed hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated."),
 Sentence("pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool."),
 Sentence("some strains of corn (botanically classified as zea mays) are cultivated specifically as popping corns."),
 Sentence("the zea mays variety everta, a special kind of flint corn, is the most common of these."),
 Sentence("p

In [45]:
# Convert text blob sentences to strings
my_sentences_str = [ str(x) for x in my_sentences ]
my_sentences_str


['popcorn (also called popped corn, popcorns, or pop-corn) is a variety of corn kernel which expands and puffs up when heated.',
 'the term also refers to the snack food produced by the expansion.',
 'it is one of the oldest snacks, with evidence of popcorn dating back thousands of years in the americas.',
 'it is commonly eaten salted, buttered, sweetened, or with artificial flavorings.',
 'a popcorn kernel strong hull contains the seed hard, starchy shell endosperm with 14–20% moisture, which turns to steam as the kernel is heated.',
 'pressure from the steam continues to build until the hull ruptures, allowing the kernel to forcefully expand, to 20 to 50 times its original size, and then cool.',
 'some strains of corn (botanically classified as zea mays) are cultivated specifically as popping corns.',
 'the zea mays variety everta, a special kind of flint corn, is the most common of these.',
 'popcorn is one of six major types of corn, alongside dent corn, flint corn, pod corn, flou

In [46]:
# Perform the TF-IDF Vectorization
tf_idf_vec = TfidfVectorizer(stop_words = 'english')
tf_idf_pop = tf_idf_vec.fit_transform(my_sentences_str)
tf_idf_pop.shape


(128, 879)

In [47]:
np.prod(tf_idf_pop.shape)

np.int64(112512)

In [48]:
tf_idf_pop


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1441 stored elements and shape (128, 879)>

In [49]:
tf_idf_pop.transpose().shape


(879, 128)

In [50]:
np.set_printoptions(precision=6, formatter=None)
tf_idf_vec.get_feature_names_out()


array(['000', '10', '100', '12', '135', '14', '15', '180', '1800s', '1848', '1885', '1890s', '1893', '1896', '1922', '1930s', '1938', '1940s', '1948', '1950', '1970', '1981', '1987', '1998', '1999',
       '19th', '20', '2001', '2007', '2011', '20th', '232', '2377', '25', '300', '356', '36', '3600', '382', '45', '450', '50', '56676', '57003', '700', '78', '8493', '930', '932', '978', 'able',
       'academy', 'accessibility', 'according', 'acetyl', 'achieved', 'acids', 'act', 'add', 'additives', 'aden', 'agents', 'agitator', 'ago', 'air', 'airy', 'allowed', 'allowing', 'alongside',
       'altered', 'amazon', 'america', 'american', 'americanisms', 'americans', 'americas', 'amounts', 'amusement', 'ancestral', 'andrew', 'anthropology', 'appeared', 'appliance', 'applied',
       'appreciable', 'approximately', 'archaeological', 'archaeologists', 'arnel', 'aroma', 'artificial', 'associated', 'association', 'attached', 'attendance', 'attention', 'attract', 'attractive',
       'attractivene

In [51]:
for w in ['use', 'used', 'uses', 'using',]:
  print(f"stem: {w} -> {TextBlob( w ).words[0].stem()}")

for w in ['use', 'used', 'uses', 'using',]:
  print(f"lemmatize: {w} -> {TextBlob( w ).words[0].lemmatize()}")

for w in ['use', 'used', 'uses', 'using',]:
  print(f"singularize: {w} -> {TextBlob( w ).words[0].singularize()}")


stem: use -> use
stem: used -> use
stem: uses -> use
stem: using -> use
lemmatize: use -> use
lemmatize: used -> used
lemmatize: uses -> us
lemmatize: using -> using
singularize: use -> use
singularize: used -> used
singularize: uses -> use
singularize: using -> using


In [52]:
# Print out results in a dataframe
tf_df = pd.DataFrame(tf_idf_pop.toarray(), columns = tf_idf_vec.get_feature_names_out())
tf_df.transpose()


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127
000,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.290129,0.0,0.283972,0.0,0.100540,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.369332,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.332144,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.170849,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.261483,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
135,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.254568,0.0,0.0,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
years,0.0,0.0,0.31138,0.0,0.0,0.0,0.000000,0.000000,0.0,0.263833,0.0,0.258234,0.0,0.091428,0.0,0.0,0.0,0.320648,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
yellow,0.0,0.0,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.441109,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.

In [53]:
# Get a data frame with the TF-IDF values sorted for document 0
df = pd.DataFrame(tf_idf_pop[0].T.todense(), index=tf_idf_vec.get_feature_names_out(), columns=["TF-IDF"])
df = df.sort_values('TF-IDF', ascending=False)
df[:5]


,TF-IDF
corn,0.513506
called,0.349884
popcorns,0.349884
puffs,0.322426
expands,0.302945


In [54]:
tf_df.T[[0]].sort_values([0], ascending=False)[:5]


,0
corn,0.513506
called,0.349884
popcorns,0.349884
puffs,0.322426
expands,0.302945


In [55]:
(
  tf_idf_pop[0]
  .T
  .todense()
)


matrix([[0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
        [0.      ],
